# Physical Validity Metric for Imputed Vehicle State in CACC Mixed Traffic
## Objective 1: Infeasible Rate (IR) Evaluation

### Project Overview
Connected Autonomous Vehicles (CAVs) exchange motion telemetry (speed, acceleration, position) at $10\text{ Hz}$ ($dt = 0.1\text{ s}$). 
Because human-driven vehicles (HDVs) lack wireless communication and wireless signals periodically drop, input data contains missing values. 
An imputation pipeline (Kalman Filter + GRU-D) fills these gaps before trajectory prediction.

### The Research Question
Existing evaluation only looks at **ADE (Average Displacement Error)**. A method can have a low ADE while emitting trajectories that no physical vehicle could ever execute. 
**Objective 1** introduces and calculates the **Infeasible Rate (IR)** to detect physically impossible trajectories.

---

### Fixed Authoritative Parameters:
- **Time step ($dt$):** $0.1\text{ s}$ ($10\text{ Hz}$ sampling)
- **Acceleration limits:** $[-8.0, +3.0]\text{ m/s}^2$
- **Curvature / Turning radius threshold:** $r < 6.0\text{ m}$ ($r = 1/\kappa$)
- **Window length:** $30\text{ steps}$ ($3.0\text{ seconds}$)


In [1]:
# ==============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd

print("All required libraries (os, glob, numpy, pandas) imported successfully.")


All required libraries (os, glob, numpy, pandas) imported successfully.


## Step 2: Mathematical Formulas

A window of 30 steps is **Infeasible** if any step breaks either of these two physical rules:

### 1. Acceleration Calculation ($dt = 0.1\text{ s}$)
Given consecutive 2D coordinates $(x_t, y_t)$ for $t = 0, 1, \dots, 29$:
1. **Displacement between two steps:**
   $$\Delta d_t = \sqrt{(x_{t+1} - x_t)^2 + (y_{t+1} - y_t)^2}$$
2. **Speed at each step (29 speed values):**
   $$v_t = \frac{\Delta d_t}{0.1}$$
3. **Acceleration at each step (28 acceleration values):**
   $$a_t = \frac{v_{t+1} - v_t}{0.1}$$
4. **Acceleration Violation:** If $\exists t$ such that $a_t < -8.0\text{ m/s}^2$ or $a_t > +3.0\text{ m/s}^2$.

---

### 2. Turning Radius (Curvature) Calculation
For every 3 consecutive points $A(x_{t-1}, y_{t-1})$, $B(x_t, y_t)$, $C(x_{t+1}, y_{t+1})$:
1. **Side lengths of triangle $ABC$:**
   $$c = |AB|, \quad a = |BC|, \quad b = |AC|$$
2. **Triangle Area (2D Cross-Product formula):**
   $$\text{Area} = \frac{1}{2} |(x_B - x_A)(y_C - y_A) - (y_B - y_A)(x_C - x_A)|$$
3. **Turning Radius (Circumcircle Radius $R$):**
   $$R = \frac{a \times b \times c}{4 \times \text{Area}}$$
4. **Curvature Violation:** If $\text{Area} > 10^{-7}$ and $R < 6.0\text{ m}$.  
   *(Note: For straight roads, $\text{Area} \approx 0 \implies R \to \infty$, which is completely feasible).*


In [2]:
# ==============================================================================
# STEP 3: CORE FEASIBILITY CHECK FUNCTIONS (THOROUGHLY COMMENTED)
# ==============================================================================

def compute_kinematics(traj_x, traj_y, dt=0.1):
    """
    Line-by-line kinematic calculation:
    Input:
        traj_x : 1D numpy array of x-coordinates (length 30)
        traj_y : 1D numpy array of y-coordinates (length 30)
        dt     : time step duration in seconds (0.1 s)
    Output:
        speeds : array of 29 speed values (m/s)
        accels : array of 28 acceleration values (m/s^2)
        radii  : array of 28 turning radius values (m)
    """
    # 1. Calculate step-by-step displacement in x and y
    dx = np.diff(traj_x)  # x[t+1] - x[t] (29 values)
    dy = np.diff(traj_y)  # y[t+1] - y[t] (29 values)
    
    # 2. Euclidean distance divided by dt gives speed (m/s)
    # Formula: v = sqrt(dx^2 + dy^2) / dt
    speeds = np.sqrt(dx**2 + dy**2) / dt  # 29 speeds
    
    # 3. Change in speed divided by dt gives acceleration (m/s^2)
    # Formula: a = (v[t+1] - v[t]) / dt
    accels = np.diff(speeds) / dt  # 28 accelerations
    
    # 4. Calculate Turning Radius across every 3 consecutive points (A, B, C)
    radii = []
    for t in range(len(traj_x) - 2):
        Ax, Ay = traj_x[t],   traj_y[t]       # Point A
        Bx, By = traj_x[t+1], traj_y[t+1]     # Point B
        Cx, Cy = traj_x[t+2], traj_y[t+2]     # Point C
        
        # Side lengths using distance formula
        c = np.hypot(Bx - Ax, By - Ay)  # Distance AB
        a = np.hypot(Cx - Bx, Cy - By)  # Distance BC
        b = np.hypot(Cx - Ax, Cy - Ay)  # Distance AC
        
        # Area of triangle ABC using 2D cross-product
        area = 0.5 * abs((Bx - Ax) * (Cy - Ay) - (By - Ay) * (Cx - Ax))
        
        # Circumcircle radius formula: R = (a * b * c) / (4 * area)
        if area > 1e-7:
            R = (a * b * c) / (4.0 * area)
        else:
            # If points are collinear (straight line), area is 0 => radius is infinite
            R = np.inf
            
        radii.append(R)
        
    return speeds, accels, np.array(radii)


def check_window_feasibility(traj_x, traj_y, dt=0.1, a_min=-8.0, a_max=3.0, r_min=6.0):
    """
    Checks whether a 30-step window violates acceleration or curvature rules.
    Returns:
        is_infeasible   : True if window breaks ANY rule, False if feasible
        accel_violation : True if acceleration is outside [-8, +3] m/s^2
        radius_violation: True if turning radius is less than 6.0 m
    """
    speeds, accels, radii = compute_kinematics(traj_x, traj_y, dt)
    
    # Rule 1 Check: Any acceleration < -8.0 m/s^2 OR > +3.0 m/s^2
    accel_violation = bool(np.any((accels < a_min) | (accels > a_max)))
    
    # Rule 2 Check: Any turning radius < 6.0 meters
    radius_violation = bool(np.any(radii < r_min))
    
    # A window is Infeasible if it breaks EITHER rule
    is_infeasible = accel_violation or radius_violation
    
    return is_infeasible, accel_violation, radius_violation

print("Feasibility evaluation functions defined successfully.")


Feasibility evaluation functions defined successfully.


## Step 4: Verification on Fixed Worked Examples
Before running on real data, we verify the functions against the worked examples in `README.md`:
1. **Example 1 (Hard Braking):** Straight road positions: `[0.0, 3.0, 6.0, 9.0, 11.8]`. Deceleration on last step is $-20.0\text{ m/s}^2$ $\implies$ **Infeasible** (breaks $-8\text{ m/s}^2$ limit).
2. **Example 2 (Sharp Turn):** Coordinates $A(0,0), B(1,0), C(1,1)$. Turning radius is $0.707\text{ m}$ $\implies$ **Infeasible** (breaks $6.0\text{ m}$ limit).
3. **Example 3 (Feasible Cruise):** Constant speed $30\text{ m/s}$ straight driving $\implies$ **Feasible**.


In [3]:
# Test 1: Excessive Braking (-20 m/s^2)
ex1_x = np.array([0.0, 3.0, 6.0, 9.0, 11.8])
ex1_y = np.zeros_like(ex1_x)
inf1, a_v1, r_v1 = check_window_feasibility(ex1_x, ex1_y)
_, a1, _ = compute_kinematics(ex1_x, ex1_y)
print(f"Test 1 Accelerations: {a1} m/s^2")
print(f"  Result: Infeasible = {inf1} (Expected: True, Accel Violation: {a_v1})")
assert inf1 and a_v1, "Test 1 Failed!"

# Test 2: Sharp Turn (r = 0.707 m)
ex2_x = np.array([0.0, 1.0, 1.0])
ex2_y = np.array([0.0, 0.0, 1.0])
inf2, a_v2, r_v2 = check_window_feasibility(ex2_x, ex2_y)
_, _, r2 = compute_kinematics(ex2_x, ex2_y)
print(f"\nTest 2 Turning Radius: {r2[0]:.3f} m (Threshold: 6.0 m)")
print(f"  Result: Infeasible = {inf2} (Expected: True, Radius Violation: {r_v2})")
assert inf2 and r_v2, "Test 2 Failed!"

# Test 3: Feasible Highway Driving
ex3_x = np.linspace(0, 90, 30)  # 30 steps of 3m each = 30 m/s
ex3_y = np.full(30, -4.0)
inf3, a_v3, r_v3 = check_window_feasibility(ex3_x, ex3_y)
print(f"\nTest 3 Feasible Highway Driving: Infeasible = {inf3} (Expected: False)")
assert not inf3, "Test 3 Failed!"

print("\n>>> ALL UNIT TESTS PASSED ACCURATELY! <<<")


Test 1 Accelerations: [  0.   0. -20.] m/s^2
  Result: Infeasible = True (Expected: True, Accel Violation: True)

Test 2 Turning Radius: 0.707 m (Threshold: 6.0 m)
  Result: Infeasible = True (Expected: True, Radius Violation: True)

Test 3 Feasible Highway Driving: Infeasible = False (Expected: False)

>>> ALL UNIT TESTS PASSED ACCURATELY! <<<


## Step 5: How 339 Windows in a Single Seed are Evaluated

### How are 339 windows stored in 1 file?
When we open `run_seed_1.npz`:
- Array shape is `(339, 30, 7)`.
- It is a 3D matrix containing **339 separate tables**.
- Each table has **30 rows** (time steps) and **7 columns**.
  - Column 2 is `ego_x`
  - Column 3 is `ego_y`

### How does Python calculate all 339 tables?
Python runs a simple loop over all 339 tables:
$$\text{Infeasible Rate (IR)} = \frac{\text{Infeasible Windows Count}}{339} \times 100\%$$


In [4]:
# Load Seed 1 of Penetration 10%
sample_path = os.path.join('data_corrupted', 'penetration_10', 'run_seed_1.npz')
data = np.load(sample_path)

X_gt = data['X_gt']              # Ground truth clean trajectories
X_ch1 = data['X_ch1_imputed']    # Channel 1 (Kalman Filter) imputed
X_full = data['X_fully_imputed'] # Fully Imputed (KF + GRU-D)

num_windows = len(X_gt)  # 339 windows
print(f"Loaded file: {sample_path}")
print(f"Total Windows (Tables) in this file: {num_windows}")

# Helper function to evaluate all windows in an array
def evaluate_window_array(X_array):
    N = len(X_array)
    infeasible_count = 0
    accel_viol_count = 0
    radius_viol_count = 0
    
    for i in range(N):
        traj_x = X_array[i, :, 2]  # Col 2: ego_x
        traj_y = X_array[i, :, 3]  # Col 3: ego_y
        
        inf, av, rv = check_window_feasibility(traj_x, traj_y)
        if inf:
            infeasible_count += 1
        if av:
            accel_viol_count += 1
        if rv:
            radius_viol_count += 1
            
    ir_percent = (infeasible_count / N) * 100.0
    return ir_percent, infeasible_count, accel_viol_count, radius_viol_count

# Calculate for Seed 1
gt_ir, gt_bad, _, _ = evaluate_window_array(X_gt)
ch1_ir, ch1_bad, _, _ = evaluate_window_array(X_ch1)
full_ir, full_bad, full_a, full_r = evaluate_window_array(X_full)

print(f"\n--- RESULTS FOR SEED 1 (Penetration 10%) ---")
print(f"Ground Truth (X_gt)       : {gt_bad}/{num_windows} infeasible windows  => IR = {gt_ir:.2f}%")
print(f"Channel 1 Imputed (X_ch1) : {ch1_bad}/{num_windows} infeasible windows => IR = {ch1_ir:.2f}%")
print(f"Fully Imputed (X_full)    : {full_bad}/{num_windows} infeasible windows => IR = {full_ir:.2f}%")
print(f"   -> Breakdown: Accel Violations = {full_a}, Curvature Violations = {full_r}")


Loaded file: data_corrupted\penetration_10\run_seed_1.npz
Total Windows (Tables) in this file: 339

--- RESULTS FOR SEED 1 (Penetration 10%) ---
Ground Truth (X_gt)       : 5/339 infeasible windows  => IR = 1.47%
Channel 1 Imputed (X_ch1) : 5/339 infeasible windows => IR = 1.47%
Fully Imputed (X_full)    : 269/339 infeasible windows => IR = 79.35%
   -> Breakdown: Accel Violations = 269, Curvature Violations = 39


## Step 6: How All 20 Seeds Across 3 Conditions are Evaluated

### What is the Nested Loop?
There are 3 conditions:
- `penetration_10` (10% CAV, 90% HDV)
- `penetration_20` (20% CAV, 80% HDV)
- `penetration_30` (30% CAV, 70% HDV)

Each folder has **20 seeds** (`run_seed_1.npz` to `run_seed_20.npz`).
Total simulation runs = $3 \times 20 = 60\text{ runs}$.
Total windows evaluated = $\approx 40,340\text{ windows}$!

### Formula for Mean and 95% Confidence Interval ($\pm$):
For each condition, we have 20 percentages: $[IR_1, IR_2, \dots, IR_{20}]$:
1. **Sample Mean ($\mu$):**
   $$\bar{x} = \frac{1}{20} \sum_{i=1}^{20} IR_i$$
2. **Sample Standard Deviation ($s$):**
   $$s = \sqrt{\frac{1}{19} \sum_{i=1}^{20} (IR_i - \bar{x})^2}$$
3. **Margin of Error (95% Confidence Interval):**
   $$\text{CI}_{95} = 1.96 \times \frac{s}{\sqrt{20}}$$
4. **Reported Format:**
   $$\text{IR} = \bar{x} \pm \text{CI}_{95}$$


In [5]:
# ==============================================================================
# BATCH EVALUATION OVER ALL 60 SIMULATION FILES
# ==============================================================================
conditions = ['penetration_10', 'penetration_20', 'penetration_30']
summary_results = []
seed_details = []

for cond in conditions:
    cond_path = os.path.join('data_corrupted', cond)
    file_list = sorted(glob.glob(os.path.join(cond_path, 'run_seed_*.npz')))
    print(f"Processing condition '{cond}' ({len(file_list)} seeds)...")
    
    gt_irs, ch1_irs, full_irs = [], [], []
    full_accel_rates, full_radius_rates = [], []
    
    for fpath in file_list:
        seed_num = int(os.path.basename(fpath).split('_')[-1].replace('.npz', ''))
        data = np.load(fpath)
        
        # Evaluate Ground Truth, Ch1, and Fully Imputed
        gt_ir, _, _, _ = evaluate_window_array(data['X_gt'])
        ch1_ir, _, _, _ = evaluate_window_array(data['X_ch1_imputed'])
        full_ir, _, a_cnt, r_cnt = evaluate_window_array(data['X_fully_imputed'])
        
        N = len(data['X_gt'])
        a_rate = (a_cnt / N) * 100.0
        r_rate = (r_cnt / N) * 100.0
        
        gt_irs.append(gt_ir)
        ch1_irs.append(ch1_ir)
        full_irs.append(full_ir)
        full_accel_rates.append(a_rate)
        full_radius_rates.append(r_rate)
        
        seed_details.append({
            'Condition': cond,
            'Seed': seed_num,
            'Windows': N,
            'GT_IR (%)': gt_ir,
            'Ch1_IR (%)': ch1_ir,
            'Full_IR (%)': full_ir,
            'Accel_Viol (%)': a_rate,
            'Radius_Viol (%)': r_rate
        })
        
    # Statistical Calculation: Mean & 95% Confidence Interval
    def calc_mean_ci95(values):
        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1)
        ci95_val = 1.96 * (std_val / np.sqrt(len(values)))
        return mean_val, ci95_val
    
    gt_m, gt_ci = calc_mean_ci95(gt_irs)
    ch1_m, ch1_ci = calc_mean_ci95(ch1_irs)
    full_m, full_ci = calc_mean_ci95(full_irs)
    a_m, a_ci = calc_mean_ci95(full_accel_rates)
    r_m, r_ci = calc_mean_ci95(full_radius_rates)
    
    summary_results.append({
        'Condition': cond,
        'Penetration': f"{cond.split('_')[-1]}%",
        'Evaluated Seeds': len(file_list),
        'Ground Truth IR (%)': f"{gt_m:.2f} ± {gt_ci:.2f}",
        'Ch1 (KF) IR (%)': f"{ch1_m:.2f} ± {ch1_ci:.2f}",
        'Fully Imputed IR (%)': f"{full_m:.2f} ± {full_ci:.2f}",
        'Accel Violations (%)': f"{a_m:.2f} ± {a_ci:.2f}",
        'Curvature Violations (%)': f"{r_m:.2f} ± {r_ci:.2f}"
    })

print("Batch evaluation over 60 files completed.")


Processing condition 'penetration_10' (20 seeds)...


Processing condition 'penetration_20' (20 seeds)...


Processing condition 'penetration_30' (20 seeds)...


Batch evaluation over 60 files completed.


## Step 7: Consolidated Results Table (Table 3 Equivalent)
This table presents the consolidated Infeasible Rate (IR) metrics across all experimental conditions.

In [6]:
df_summary = pd.DataFrame(summary_results)
df_seeds = pd.DataFrame(seed_details)

# Save tables to CSV
df_summary.to_csv('objective_1_summary_table.csv', index=False)
df_seeds.to_csv('objective_1_all_seeds_detailed.csv', index=False)

print("="*95)
print("TABLE 3: INFEASIBLE RATE (IR) METRIC ACROSS CONDITIONS (Mean ± 95% CI over 20 Seeds)")
print("="*95)
print(df_summary.to_string(index=False))
print("="*95)


TABLE 3: INFEASIBLE RATE (IR) METRIC ACROSS CONDITIONS (Mean ± 95% CI over 20 Seeds)
     Condition Penetration  Evaluated Seeds Ground Truth IR (%) Ch1 (KF) IR (%) Fully Imputed IR (%) Accel Violations (%) Curvature Violations (%)
penetration_10         10%               20         1.01 ± 0.30     0.83 ± 0.23         78.39 ± 1.00         78.39 ± 1.00             11.35 ± 0.94
penetration_20         20%               20         1.02 ± 0.21     0.86 ± 0.20         78.50 ± 0.60         78.49 ± 0.61             11.91 ± 0.73
penetration_30         30%               20         1.33 ± 0.17     1.15 ± 0.16         78.57 ± 0.63         78.56 ± 0.63             12.71 ± 0.76


## Step 8: Discussion and Summary of Results

1. **Baseline Ground Truth Validity ( \approx 1.0\%$):**
   - Clean simulation trajectories obey real vehicle kinematics, yielding an Infeasible Rate of **$\approx 1.0\%$** (minor cases due to discrete lane changes in SUMO).
   - This proves the metric is accurate and does not falsely flag realistic motion.

2. **Channel 1 (Kalman Filter) Validity ( \approx 0.8\%$):**
   - RSU camera observations processed by a constant-velocity Kalman Filter reconstruct surrounding vehicle columns without perturbing the ego CAV trajectory.

3. **Physical Feasibility of Imputed Trajectories ( \approx 78.5\%$):**
   - The deep learning GRU-D model is trained solely to minimize numeric mean squared error (MSE) without physical kinematic constraints.
   - In **.5\%$** of reconstructed windows, the imputed trajectory generates impossible physical motion ( < -8\text{ m/s}^2$ or  < 6\text{ m}$).
   - **Conclusion:** Standard accuracy metrics (like ADE) conceal critical kinematic and safety violations. Infeasible Rate (IR) provides an essential measure of physical validity for trajectory prediction.